# Reconocer dígitos: entrenamiento de la CNN

Entrena una red convolucional pequeña sobre MNIST y exporta los pesos a JSON para que la demo del navegador (`index.html`) pueda calcular la predicción sin servidor.

**Cómo ejecutarlo:** desde la carpeta `numeros/`, para que las rutas `modelo/...` apunten bien. El entrenamiento de 30 épocas tarda bastante en CPU.

In [ ]:
import json, os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import numpy as np, keras
from keras import layers

keras.utils.set_random_seed(0)

## Datos

MNIST tiene 60 000 imágenes de entrenamiento y 10 000 de test, de 28×28 píxeles en escala de grises. Los píxeles se pasan de [0, 255] a [0, 1] y se añade el canal de color, que la CNN necesita.

In [ ]:
(xtr, ytr), (xte, yte) = keras.datasets.mnist.load_data()
xtr = xtr[..., None].astype("float32") / 255
xte = xte[..., None].astype("float32") / 255
print(xtr.shape, xte.shape)

## Modelo

Antes de las capas de convolución hay aumentos de datos (traslación, rotación y zoom leves) que solo se aplican durante el entrenamiento. Después, dos bloques convolución + max-pooling, y dos capas densas con dropout para reducir el sobreajuste.

In [ ]:
model = keras.Sequential([
    keras.Input((28, 28, 1)),
    layers.RandomTranslation(0.08, 0.08), layers.RandomRotation(0.04), layers.RandomZoom(0.08),
    layers.Conv2D(32, 3, activation="relu"), layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"), layers.MaxPooling2D(),
    layers.Flatten(), layers.Dropout(0.3), layers.Dense(128, activation="relu"), layers.Dropout(0.3), layers.Dense(10),
])
model.compile(keras.optimizers.Adam(1e-3),
              keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=["accuracy"])
model.summary()

## Entrenamiento

30 épocas con batch de 128. Se reserva un 5 % de entrenamiento para validación y, si la validación se estanca, `ReduceLROnPlateau` reduce a la mitad la tasa de aprendizaje.

In [ ]:
model.fit(xtr, ytr, epochs=30, batch_size=128, validation_split=0.05, verbose=2,
          callbacks=[keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2)])

## Evaluación sobre el test

In [ ]:
loss, acc = model.evaluate(xte, yte, verbose=0)
print("TEST ACC", acc)

## Exportación

- `modelo/cnn_mnist.json`: pesos de las capas convolucionales y densas, redondeados a 4 decimales. Es lo que carga `index.html`.
- `modelo/cnn_mnist.keras`: el modelo completo, para usarlo desde Python.
- `modelo/ejemplos_test.npy` y `modelo/ejemplos_y.npy`: 20 imágenes de test y sus etiquetas.
- `modelo/ref_logits.npy`: salidas del modelo para esas 20 imágenes. Sirven para comprobar que la implementación en JavaScript da el mismo resultado.

In [ ]:
convs = [l for l in model.layers if isinstance(l, layers.Conv2D)]
dense = [l for l in model.layers if isinstance(l, layers.Dense)]
r = lambda a: np.round(a, 4).tolist()
out = {"test_acc": float(acc),
       "conv1": {"w": r(convs[0].kernel.numpy()), "b": r(convs[0].bias.numpy())},
       "conv2": {"w": r(convs[1].kernel.numpy()), "b": r(convs[1].bias.numpy())},
       "fc1": {"w": r(dense[0].kernel.numpy()), "b": r(dense[0].bias.numpy())},
       "fc2": {"w": r(dense[1].kernel.numpy()), "b": r(dense[1].bias.numpy())}}
json.dump(out, open("modelo/cnn_mnist.json", "w"), separators=(",", ":"))
model.save("modelo/cnn_mnist.keras")
np.save("modelo/ejemplos_test.npy", xte[:20, ..., 0]); np.save("modelo/ejemplos_y.npy", yte[:20])
# referencia para verificar el JS
np.save("modelo/ref_logits.npy", model.predict(xte[:20], verbose=0))